# 03 · Avaliação automática: Pydantic + tool selection + LLM-as-judge

**Objetivo:** avaliar o agente **sem inspeção manual**, em camadas — das mais baratas para a mais cara:

| Camada | Avaliador | Custo | O que pega |
|---|---|---|---|
| Estrutural | `schema_valido` (Pydantic) | grátis | receita fora do contrato do frontend |
| Tool selection | `tools_exatas` (0/1) e `tools_f1` (parcial) | grátis | tool faltando (qualidade) ou sobrando (custo) |
| Regra de negócio | `aproveitamento_prioritarios` | grátis | ignorou o que vai vencer |
| Qualidade | `juiz_qualidade` (LLM-as-judge) | 1 chamada de LLM | receita sem sentido, item vencido, não atende ao pedido |

### Como implementar o juiz? (decisão)

| Opção | Esforço | Quando usar |
|---|---|---|
| **Função Python custom passada no `evaluate()`** ✅ | mínimo: uma função + um schema Pydantic | **escolhida** — controle total da rubrica, roda offline contra o dataset, nota vira *feedback* no LangSmith |
| `openevals.create_llm_as_judge` | baixo, mais uma dependência | prompts de juiz prontos (correctness, conciseness…) |
| Online evaluator na UI do LangSmith | sem código | avaliar **tráfego de produção** continuamente (próximo passo natural) |

A função custom **é** um evaluator do LangSmith: o `evaluate()` aceita qualquer função `(inputs, outputs, reference_outputs) -> {"key", "score"}`.
Então não é "um ou outro" — é a forma mais simples de ter os dois.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))  # adiciona a raiz do projeto ao path

from receitas import config

In [3]:
import json
import pandas as pd
from receitas.dataset import carregar_ou_criar_golden
from receitas import avaliadores as av

exemplos = carregar_ou_criar_golden()
por_id = {e["outputs"]["id_cenario"]: e for e in exemplos}

## 1. Avaliadores determinísticos (grátis e instantâneos)

In [6]:
ref = por_id["M2"]["outputs"]
print("Tools esperadas em M2:", ref["tools_esperadas"])

casos = {
    "agente exato":            ["verificar_validade", "criar_receita"],
    "chamou tool a mais":      ["check_geladeira", "verificar_validade", "criar_receita"],
    "esqueceu a validade":     ["check_geladeira", "criar_receita"],
    "nenhuma tool":            [],
}
pd.DataFrame([{
    "caso": nome,
    "tools_exatas": av.tools_exatas({"tools_chamadas": t}, ref)["score"],
    "tools_f1": av.tools_f1({"tools_chamadas": t}, ref)["score"],
    "comentário": av.tools_exatas({"tools_chamadas": t}, ref)["comment"],
} for nome, t in casos.items()])

Tools esperadas em M2: ['verificar_validade', 'criar_receita']


,caso,tools_exatas,tools_f1,comentário
0,agente exato,1,1.0,ok
1,chamou tool a mais,0,0.8,faltaram=[] sobraram=['check_geladeira']
2,esqueceu a validade,0,0.5,faltaram=['verificar_validade'] sobraram=['che...
3,nenhuma tool,0,0.0,"faltaram=['criar_receita', 'verificar_validade..."


In [18]:
receita_ok = {"nome": "Sopa de legumes", "descricao": "Sopa caseira para aproveitar a geladeira.",
              "ingredientes": ["2 chuchus", "1 cebola"], "modo_preparo": ["Pique", "Cozinhe 20 min"],
              "tempo_preparo_min": 30, "porcoes": 2}
receita_quebrada = {**receita_ok, "porcoes": 0, "modo_preparo": []}

print(av.schema_valido({"receita": receita_ok}, ref))
print(av.schema_valido({"receita": receita_quebrada}, ref))

{'key': 'schema_valido', 'score': 1}
{'key': 'schema_valido', 'score': 0, 'comment': '2 validation errors for Receita\nmodo_preparo\n  List should have at least 2 items after validation, not 0 [type=too_short, input_value=[], input_type=list]\n    For further information visit https://errors.pydantic.dev/2.13/v/too_short\nporcoes\n  Input should be greater than 0 [type=greater_than, input'}


## 2. Calibrar o juiz antes de confiar nele

Um LLM-as-judge também erra. Antes de usar a nota dele para decidir qualquer coisa, testamos se ele **separa o bom do ruim**:
pegamos uma resposta real do agente e criamos versões **sabidamente ruins** (corrupções determinísticas):

1. receita usando um item **vencido**;
2. receita com ingredientes que **não estão na geladeira**;
3. modo de preparo **incoerente**.

Se o juiz não der nota menor para as versões ruins, a rubrica precisa melhorar.

In [7]:
from receitas.agente import criar_alvo

exemplo = por_id["M2"]
boa = criar_alvo(modelo=config.MODELO_FRONTEIRA)(exemplo["inputs"])
print("Resposta do agente:", boa["resposta"])
print("Receita:", json.dumps(boa["receita"], ensure_ascii=False)[:500])

Resposta do agente: Sugiro "Bacalhau ao Forno com Legumes e Creme de Pesto", que aproveita todos os itens que vencem logo: repolho, pesto, bacalhau dessalgado, chuchu, milho verde, cream cheese e laranja. Assim, você evita desperdício e garante um almoço saboroso e completo. Receita já disponível no app!
Receita: {"nome": "Bacalhau ao Forno com Legumes e Creme de Pesto", "descricao": "Um prato saboroso e prático que aproveita bacalhau, legumes frescos e um delicioso creme de pesto, perfeito para um almoço especial.", "ingredientes": ["300 g de bacalhau dessalgado", "1/2 repolho pequeno fatiado", "1 chuchu médio em cubos", "1 lata de milho verde escorrido", "2 colheres de sopa de pesto", "100 g de cream cheese", "1/2 laranja (suco)", "1 abobrinha média em cubos", "1/2 xícara de queijo parmesão ralado", "2


In [8]:
vencido = exemplo["outputs"]["ingredientes_vencidos"][0]
r = boa["receita"]
corrompidas = {
    "boa (original)": r,
    "usa item vencido": {**r, "ingredientes": r["ingredientes"] + [f"300 g de {vencido}"]},
    "ingredientes inexistentes": {**r, "ingredientes": ["200 g de trufas negras", "50 g de caviar", "1 lagosta"]},
    "preparo incoerente": {**r, "modo_preparo": ["Sirva imediatamente.", "Leve ao forno por 6 horas.", "Corte os ingredientes."],
                           "tempo_preparo_min": 2},
}

notas = []
for nome, receita in corrompidas.items():
    avaliacao = av.julgar(exemplo["inputs"], {"receita": receita, "resposta": boa["resposta"]}, exemplo["outputs"])
    notas.append({"versão": nome, "nota (0-1)": avaliacao.nota(), **avaliacao.model_dump(exclude={"raciocinio"})})
calibracao = pd.DataFrame(notas)
display(calibracao)

nota_boa = calibracao.iloc[0]["nota (0-1)"]
print("Juiz separa bom de ruim?", all(calibracao.iloc[1:]["nota (0-1)"] < nota_boa))

,versão,nota (0-1),uso_da_geladeira,prioriza_vencimento,coerencia_preparo,atende_pedido
0,boa (original),1.000,5,5,5,5
1,usa item vencido,0.750,3,4,5,4
2,ingredientes inexistentes,0.438,1,5,2,3
3,preparo incoerente,0.812,5,5,2,5


Juiz separa bom de ruim? True


## 3. Experimento 1 — prompt `v1` (baseline de prompt)

`evaluate()` roda o agente em **todos os exemplos do dataset congelado** e aplica todos os avaliadores.
O resultado vira um **experimento** no LangSmith (aba *Datasets & Experiments*), comparável lado a lado com os próximos.

Usamos o modelo intermediário aqui para baratear; o notebook 05 faz a comparação entre modelos.

In [9]:
from langsmith import evaluate
from receitas.tendencia import resultados_para_df, registrar_no_historico

MODELO_DO_TESTE = config.MODELO_INTERMEDIARIO

resultados_v1 = evaluate(
    criar_alvo(modelo=MODELO_DO_TESTE, versao_prompt="v1"),
    data=config.NOME_DATASET,
    evaluators=av.AVALIADORES_PADRAO,
    experiment_prefix=f"prompt-v1-{MODELO_DO_TESTE}",
    metadata={"versao_prompt": "v1", "modelo": MODELO_DO_TESTE},
    max_concurrency=4,
)
tabela_v1 = resultados_para_df(resultados_v1)
registrar_no_historico(tabela_v1, rotulo=f"prompt-v1/{MODELO_DO_TESTE}")
print(resultados_v1.experiment_name)

View the evaluation results for experiment: 'prompt-v1-gpt-4.1-mini-d25d7db4' at:
(link privado do workspace LangSmith)




0it [00:00, ?it/s]

prompt-v1-gpt-4.1-mini-d25d7db4


## 4. Experimento 2 — prompt `v2_enxuto` (teste de regressão)

Cenário realista: alguém propõe um prompt menor "para economizar tokens". Antes de ir para produção, roda contra o **mesmo** dataset.

In [9]:
resultados_v2 = evaluate(
    criar_alvo(modelo=MODELO_DO_TESTE, versao_prompt="v2_enxuto"),
    data=config.NOME_DATASET,
    evaluators=av.AVALIADORES_PADRAO,
    experiment_prefix=f"prompt-v2_enxuto-{MODELO_DO_TESTE}",
    metadata={"versao_prompt": "v2_enxuto", "modelo": MODELO_DO_TESTE},
    max_concurrency=4,
)
tabela_v2 = resultados_para_df(resultados_v2)
registrar_no_historico(tabela_v2, rotulo=f"prompt-v2_enxuto/{MODELO_DO_TESTE}")

View the evaluation results for experiment: 'prompt-v2_enxuto-gpt-4.1-mini-91996a6c' at:
(link privado do workspace LangSmith)




0it [00:00, ?it/s]

In [10]:
METRICAS = ["juiz_qualidade", "tools_exatas", "tools_f1", "schema_valido", "aproveitamento_prioritarios", "custo_usd", "latencia_s"]

comparacao = pd.DataFrame({
    "v1": tabela_v1[METRICAS].mean(),
    "v2_enxuto": tabela_v2[METRICAS].mean(),
}).round(4)
comparacao["delta"] = (comparacao["v2_enxuto"] - comparacao["v1"]).round(4)

# Marca regressão quando a qualidade cai mais de 0,05
LIMITE_QUEDA = 0.05
comparacao["regressão?"] = [
    "SIM" if m in METRICAS[:5] and d < -LIMITE_QUEDA else "" for m, d in zip(comparacao.index, comparacao["delta"])
]
display(comparacao)
print("Compare lado a lado na UI do LangSmith:", resultados_v2.comparison_url)

,v1,v2_enxuto,delta,regressão?
juiz_qualidade,0.9241,0.8527,-0.0714,SIM
tools_exatas,0.4375,0.5625,0.1250,
tools_f1,0.8446,0.9066,0.0620,
schema_valido,0.9286,0.8571,-0.0715,SIM
aproveitamento_prioritarios,0.9000,1.0000,0.1000,
custo_usd,0.0032,0.0024,-0.0008,
latencia_s,7.5025,6.0600,-1.4425,


Compare lado a lado na UI do LangSmith: (link privado do workspace LangSmith)


### Os "erros conhecidos" continuam corrigidos?

Quebrando a nota pelo histórico de cada caso: se a nova versão vai mal justamente nos casos que já foram **erro**, é regressão de algo que já tinha sido consertado.

In [11]:
pd.concat([tabela_v1.assign(prompt="v1"), tabela_v2.assign(prompt="v2_enxuto")]).pivot_table(
    index="historico_status", columns="prompt", values=["juiz_qualidade", "tools_exatas"], aggfunc="mean"
).round(3)

juiz_qualidade           tools_exatas          
prompt                       v1 v2_enxuto           v1 v2_enxuto
historico_status                                                
acerto                    0.993     0.778        0.500     0.500
erro                      0.800     0.988        0.333     0.667

## 5. Contador estatístico: a qualidade está melhorando ou piorando?

Cada `evaluate()` acima foi registrado em `data/historico_avaliacoes.csv`. Com a série de notas em ordem cronológica,
`analisar_tendencia()` combina duas ideias simples:

1. **Janelas:** média das últimas N execuções vs. as N anteriores (com margem para ignorar ruído).
2. **Contador de Mann-Kendall:** para cada par de execuções (antes, depois), soma **+1** se a nota subiu e **−1** se caiu.
   `S > 0` → tendência de alta. Robusto a outliers, porque só olha o *sinal* da diferença.

O veredito só é "melhorando"/"piorando" quando **as duas** concordam. Rode este notebook (ou o 05) várias vezes ao longo do tempo
e a série cresce; com 2–3 execuções o veredito ainda é fraco — isso é esperado.

In [12]:
from receitas.tendencia import carregar_historico, analisar_tendencia

historico = carregar_historico()
por_execucao = (historico.groupby(["registrado_em", "rotulo"], as_index=False)["juiz_qualidade"].mean()
                .sort_values("registrado_em"))
display(por_execucao)

tendencia = analisar_tendencia(por_execucao["juiz_qualidade"].tolist(), janela=3)
{k: v for k, v in tendencia.items() if k != "media_movel"}

,registrado_em,rotulo,juiz_qualidade
0,2026-10-02 14:57:10,prompt-v1/gpt-4.1-mini,0.821500
1,2026-10-02 15:02:29,prompt-v1/gpt-4.1-mini,0.924143
2,2026-10-02 15:05:18,prompt-v2_enxuto/gpt-4.1-mini,0.852714


{'veredito': 'estável',
 'n': 3,
 'media_janela_anterior': 0.924,
 'media_janela_recente': 0.853,
 'delta': -0.071,
 'mann_kendall_S': 1,
 'tau': 0.33}

## Resumo

- **Pydantic** garante o contrato; **tool selection** mede custo e acerto de rota sem LLM; o **juiz** cobre o que só um leitor perceberia.
- O juiz foi **calibrado** antes de ser usado como régua (e o modelo do juiz é fixo em todos os experimentos).
- Toda mudança de prompt/modelo passa pelo mesmo dataset congelado → regressão vira um número, não uma impressão.

**Próximo:** notebook 04 (custos e latência por tool) e 05 (escolha de modelo e roteamento).